In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install -U dagshub mlflow skops --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 59.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━

In [4]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-2-IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8b38fce8-a9b9-481c-a8e6-14c4dea695e8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d501b00fe1c84b9f63feeaebe28bec10a9ef3a6bf68e3ef8a9354c9b4dba9850




Accessing as izere23

Initialized MLflow to track repo "izere23/Assignment-2-IEEE-CIS-Fraud-Detection"

Repository izere23/Assignment-2-IEEE-CIS-Fraud-Detection initialized!

In [5]:

print('MLflow tracking URI:', mlflow.get_tracking_uri())

MLflow tracking URI: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow


In [3]:
import mlflow
import os

os.environ['MLFLOW_TRACKING_USERNAME'] = 'izere23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '7772a15bc3f28c6d5c30851893caed7ba72ecd67'

mlflow.set_tracking_uri("https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow")

In [15]:
best_run_id = "a18c6c695edd4d3790398f11d4d76e23"

In [21]:
best_pipeline = mlflow.sklearn.load_model(
    "models:/xgboost/1"
)

print("Pipeline loaded!")
print(type(best_pipeline))

Pipeline loaded!
<class 'sklearn.pipeline.Pipeline'>


In [24]:
test_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")

test_identity.columns = test_identity.columns.str.replace("-", "_")

test_df = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)

transaction_ids = test_df["TransactionID"]

In [25]:
kaggle_predictions = best_pipeline.predict_proba(test_df)[:, 1]

print(kaggle_predictions[:10])

[0.00228612 0.00684125 0.01280792 0.00166245 0.00476376 0.0055559
 0.01257524 0.02262143 0.00076605 0.00997905]


In [26]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud": kaggle_predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.shape)
print(submission.head())

(506691, 2)
   TransactionID   isFraud
0        3663549  0.002286
1        3663550  0.006841
2        3663551  0.012808
3        3663552  0.001662
4        3663553  0.004764
